# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library. The dataset describes analysis results of protein abundance, modifications, and sequences from mass spectrometry on human extracellular vesicles derived from stimulated mast cells.

### Dataset Source
The dataset source is provided via a Croissant schema URL and follows the [Croissant standard](https://mlcommons.github.io/croissant/schema.html) for metadata description and reproducible data loading.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This enables programmatic access to metadata and records for further exploration and processing.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview

Review available record sets, their `@id`s, and field structures. Each record set and its fields are uniquely identified by their `@id` per the Croissant specification to ensure reproducibility and clarity when referencing elements.

In [ ]:
# List all available record sets and their field IDs.
print("Record Sets in the dataset:")
record_set_ids = []

for record_set in metadata.record_sets:
    print(f"- @id: {record_set['@id']}, name: {record_set.get('name', '')}")
    record_set_ids.append(record_set['@id'])
    print("  Fields (by @id):")
    for field in record_set['field']:
        print(f"    - @id: {field['@id']}, name: {field.get('name', '')}, type: {field.get('dataType', '')}")
    print()

# If you want to look at fields available for the first record set in detail, uncomment below
# if record_set_ids:
#     rs = [rs for rs in metadata.record_sets if rs['@id'] == record_set_ids[0]][0]
#     for f in rs['field']:
#         print(f)


## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. The primary record set for the main protein matrix is typically named or described accordingly—for this dataset, we'll extract data from the main table record set.


In [ ]:
# Extract data from each record set by their @id
dataframes = {}
print("Loading data into pandas DataFrames:")
for record_set_id in record_set_ids:
    print(f"  - Loading {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"    WARNING: No data found for {record_set_id}")

# Show columns in the primary record set (update record_set_id as desired)
if dataframes:
    main_record_set_id = record_set_ids[0]  # Take first as main; adjust as needed
    print(f'Columns in DataFrame for {main_record_set_id}:')
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No data extracted.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.


In [ ]:
# Choose numeric and grouping fields based on previous column listing (use their @id/column name)
# If main_record_set_id not set, please re-run previous cells
if dataframes:
    df = dataframes[main_record_set_id].copy()

    # Guess likely numeric field name/column based on proteomic conventions. Override as desired.
    # E.g., typical columns: 'cr:total_abundance', 'cr:coverage_percent', 'cr:peptide_count'
    numeric_field = None
    for col in df.columns:
        if 'abundance' in col.lower() or 'coverage' in col.lower() or 'peptide' in col.lower() or 'mw' in col.lower() or 'intensity' in col.lower():
            numeric_field = col
            break

    group_field = None
    for col in df.columns:
        if 'accession' in col.lower() or 'gene' in col.lower() or 'protein' in col.lower() or 'category' in col.lower():
            group_field = col
            break

    if numeric_field:
        print(f"Selected numeric field: {numeric_field}")

        # Try to ensure field is numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

        # Filter for values greater than a threshold
        threshold = df[numeric_field].quantile(0.5)  # e.g. 50th percentile as example
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (median): {len(filtered_df)}/{len(df)} rows")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field and group_field in filtered_df.columns:
            print(f"\nGrouping by '{group_field}' for mean {numeric_field}:")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No DataFrame to analyze.")

## 5. Visualization

Visualize data distributions and relationships between fields. For example, plot the distribution of a main abundance or intensity field and show grouped means by protein/gene/accession if available.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(10,4))
    # Histogram of the numeric field
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    # If grouped means exist, plot the top 20 groups
    if group_field and 'grouped_df' in locals():
        plt.figure(figsize=(12,4))
        top_groups = grouped_df.head(20)
        sns.barplot(data=top_groups, x=group_field, y=numeric_field, palette='viridis')
        plt.title(f"Mean {numeric_field} by {group_field} (top 20)")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No data or numeric field available for visualization.")

## 6. Conclusion

In this notebook, we loaded, explored, processed, and visualized the FAIR^2 mass spectrometry dataset describing protein abundance and characteristics for extracellular vesicle preparations from human mast cells.

- **Record sets and field structure** were listed by their Croissant `@id`.
- **DataFrames** were loaded per record set using `mlcroissant`, enabling easy programmatic access.
- **Typical analysis** included filtering proteins by abundance and normalizing numeric fields, as well as grouping and visualizing the results.

For further work, consider integrating this pipeline with downstream statistical analyses, or extending the visualization to other features (e.g. peptide counts or coverage %), always referencing columns by their Croissant `@id` from the schema.
